# Sequential to Functional Assignment

The Keras sequential model is one of the easiest ways to create a model. However, by using it, one can not create all kinds of models. For example, a model that has two output heads (such models is very common in object detection). 


In this assignment, you have to implement a function that takes and sequential model and converts it to a functional model.

### Maximum Points: 30

| Section | Problem | Points |
|:------:|:--------|:------:|
|   1   | Sequential to Functional Model | 30 | 

## Sequential to Functional Model [30 points]

You must complete the function `sequential_to_functional_model` that takes a sequential model and input shape and returns a functional model with the same layers and weights.

In [1]:
import tensorflow as tf
import keras

In [2]:
from typing import Any

activationKeyword = "activation"
filtersKeyword = "filters"
kernelsKeyword = "kernel_size"
nameKeyword = "name"
paddingKeyword = "padding"
unitsKeyword = "units"


def parseConv2DLayer_sequential_to_functional(conv2d_layer_dict: dict[str, Any]):
    if nameKeyword in conv2d_layer_dict:
        name = conv2d_layer_dict[nameKeyword]
    if filtersKeyword in conv2d_layer_dict:
        filters = conv2d_layer_dict[filtersKeyword]
    if paddingKeyword in conv2d_layer_dict:
        padding = conv2d_layer_dict[paddingKeyword]
    if activationKeyword in conv2d_layer_dict:
        activation = conv2d_layer_dict[activationKeyword]
    if kernelsKeyword in conv2d_layer_dict:
        kernel = conv2d_layer_dict[kernelsKeyword]

    return keras.layers.Conv2D(
        filters=filters,
        padding=padding,
        name=name,
        activation=activation,
        kernel_size=kernel,
    )


def parseMaxPool2DLayer_sequential_to_functional(maxPool2d_layer_dict: dict[str, Any]):
    if nameKeyword in maxPool2d_layer_dict:
        name = maxPool2d_layer_dict[nameKeyword]

    return keras.layers.MaxPool2D(name=name)


def parseFlattenLayer_sequential_to_functional():
    return keras.layers.Flatten()


def parseDenseLayer_sequential_to_functional(dense_layer_dict: dict[str, Any]):
    if nameKeyword in dense_layer_dict:
        name = dense_layer_dict[nameKeyword]
    if unitsKeyword in dense_layer_dict:
        units = dense_layer_dict[unitsKeyword]
    if activationKeyword in dense_layer_dict:
        activation = dense_layer_dict[activationKeyword]

    return keras.layers.Dense(units=units, name=name, activation=activation)

In [3]:
def sequential_to_functional_model(
    sequential_model: keras.Sequential, input_shape: tuple
):
    """
    This function will take Keras's sequential model and input shape and
    returns the functional model, which has the same layers and the same
    number of weights.

    sequential_model (keras.engine.sequential.Sequential): Keras sequential model

    input_shape (tuple): Input shape of the model without batch dimension.
    For example, if the first layer is a dense layer and the number of input
    features is four, then the input shape is (4,)

    return (keras.engine.functional.Functional): returns a functional model
    """

    input = keras.Input(shape=input_shape, name="input")
    funcLayer = None
    for layer in sequential_model.layers:
        if isinstance(layer, keras.layers.Conv2D):
            if funcLayer is None:
                funcLayer = parseConv2DLayer_sequential_to_functional(
                    layer.get_config()
                )(input)
            else:
                funcLayer = parseConv2DLayer_sequential_to_functional(
                    layer.get_config()
                )(funcLayer)
        elif isinstance(layer, keras.layers.MaxPool2D):
            if funcLayer is None:
                funcLayer = parseMaxPool2DLayer_sequential_to_functional(
                    layer.get_config()
                )(input)
            else:
                funcLayer = parseMaxPool2DLayer_sequential_to_functional(
                    layer.get_config()
                )(funcLayer)
        elif isinstance(layer, keras.layers.Flatten):
            if funcLayer is None:
                funcLayer = parseFlattenLayer_sequential_to_functional()(input)
            else:
                funcLayer = parseFlattenLayer_sequential_to_functional()(funcLayer)
        elif isinstance(layer, keras.layers.Dense):
            if funcLayer is None:
                funcLayer = parseDenseLayer_sequential_to_functional(
                    layer.get_config()
                )(input)
            else:
                funcLayer = parseDenseLayer_sequential_to_functional(
                    layer.get_config()
                )(funcLayer)

    functional_model = keras.Model(
        inputs=input, outputs=funcLayer, name="functional_model"
    )

    ###
    ### YOUR CODE HERE
    ###
    return functional_model

**Test your code before submitting it using the code cell below.**

**For the given input:**

```python
sequential_model = model = tf.keras.Sequential(
    [   
        tf.keras.layers.Conv2D(filters=5,kernel_size=3,padding="valid",activation='relu',name="conv_3x3"),
        tf.keras.layers.MaxPool2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(2, activation="relu", name='Dense_1'),
        tf.keras.layers.Dense(4, name="Dense_3"),
    ],
    name = "sequential_model"
)  

input_shape = (10,10,1)

functional_model = sequential_to_functional_model(sequential_model, 
                                                  input_shape)
functional_model.summary()
print("Model Type: {}, number_params: {}".format(type(functional_model), 
                                                functional_model.count_params()))
```

**Output:**
```
Model: "functional_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
=================================================================
 input (InputLayer)          [(None, 10, 10, 1)]       0         
                                                                 
 conv_3x3 (Conv2D)           (None, 8, 8, 5)           50        
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 4, 4, 5)          0         
 2D)                                                             
                                                                 
 flatten_1 (Flatten)         (None, 80)                0         
                                                                 
 Dense_1 (Dense)             (None, 2)                 162       
                                                                 
 Dense_3 (Dense)             (None, 4)                 12        
                                                                 
=================================================================
Total params: 224
Trainable params: 224
Non-trainable params: 0
_________________________________________________________________
Model Type: <class 'keras.engine.functional.Functional'>, number_params: 224
```

**Note that for the given arguments, the outout model type must be `<class 'keras.engine.functional.Functional'>`, and the number of params must be `224`.** 

In [4]:
sequential_model = model = tf.keras.Sequential(
    [
        tf.keras.layers.Conv2D(
            filters=5,
            kernel_size=3,
            padding="valid",
            activation="relu",
            name="conv_3x3",
        ),
        tf.keras.layers.MaxPool2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(2, activation="relu", name="Dense_1"),
        tf.keras.layers.Dense(4, name="Dense_3"),
    ],
    name="sequential_model",
)

input_shape = (10, 10, 1)

functional_model = sequential_to_functional_model(sequential_model, input_shape)
functional_model.summary()
print(
    "Model Type: {}, number_params: {}".format(
        type(functional_model), functional_model.count_params()
    )
)

Model: "functional_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 10, 10, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_3x3 (Conv2D)               │ (None, 8, 8, 5)        │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 4, 4, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_1 (Dense)                 │ (None, 2)              │           162 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_3 (Dense)                 │ (None, 4)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224 (896.00 B)

 Trainable params: 224 (896.00 B)

 Non-trainable params: 0 (0.00 B)

Model Type: <class 'keras.src.models.functional.Functional'>, number_params: 224


In [5]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###